<a href="https://colab.research.google.com/github/Asritha0507/ML-Market-Basket-Analysis/blob/main/08_Data_Augmentation_and_Robustness_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import os
import gc

base_path = "/content/drive/MyDrive/ML_Market_Basket_Analysis"

sampled_path = f"{base_path}/Datasets/Augmented_Training_Data"
fixed_path = f"{base_path}/Datasets/Augmented_Labeled_Candidates_Fixed"

print("Augmented training folder:", os.path.exists(sampled_path))
print("Saved training chunks:",
      len([f for f in os.listdir(sampled_path) if f.endswith(".parquet")]))
print("Saved labeled chunks:",
      len([f for f in os.listdir(fixed_path) if f.endswith(".parquet")]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Augmented training folder: True
Saved training chunks: 20
Saved labeled chunks: 20


In [ ]:
engineered_path = f"{base_path}/Datasets/Engineered_Features"

engineered_files = sorted([
    f for f in os.listdir(engineered_path)
    if f.endswith(".parquet")
])

print("Engineered feature chunks:", len(engineered_files))

sample = pd.read_parquet(
    os.path.join(engineered_path, engineered_files[0])
)

print("Sample shape:", sample.shape)
print("Columns:")
print(list(sample.columns))

del sample
gc.collect()

Engineered feature chunks: 99
Sample shape: (735366, 21)
Columns:
['order_id', 'user_id', 'order_number', 'product_id', 'in_next_basket', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'times_purchased_before', 'first_purchase_before', 'last_purchase_before', 'recency_before', 'purchase_span_before', 'was_previously_purchased', 'aisle_id', 'department_id', 'previous_orders', 'avg_basket_before', 'max_basket_before', 'total_items_before', 'order_progress']


0

In [ ]:
import psutil

ram = psutil.virtual_memory()

print("Total RAM:", round(ram.total / (1024**3), 2), "GB")
print("Available RAM:", round(ram.available / (1024**3), 2), "GB")
print("Used RAM:", round(ram.used / (1024**3), 2), "GB")

Total RAM: 12.67 GB
Available RAM: 11.25 GB
Used RAM: 1.16 GB


In [ ]:
aug_files = sorted([
    os.path.join(sampled_path, f)
    for f in os.listdir(sampled_path)
    if f.endswith(".parquet")
])

aug_test = pd.read_parquet(aug_files[0])

print("First augmented chunk shape:", aug_test.shape)
print("Columns:", list(aug_test.columns))
print("Target orders:", aug_test["target_order_id"].nunique())
print("Positive rows:", int(aug_test["in_next_basket"].sum()))
print("Memory:", round(aug_test.memory_usage(deep=True).sum() / (1024**2), 2), "MB")

del aug_test
gc.collect()

First augmented chunk shape: (834755, 4)
Columns: ['target_order_id', 'target_user_id', 'product_id', 'in_next_basket']
Target orders: 5000
Positive rows: 39965
Memory: 19.9 MB


0

In [ ]:
orders_path = f"{base_path}/Datasets/orders.csv"

orders = pd.read_csv(orders_path)

aug_target_ids = pd.concat(
    [pd.read_parquet(f, columns=["target_order_id"]) for f in aug_files],
    ignore_index=True
)["target_order_id"].drop_duplicates().astype("int64")

aug_targets = orders[
    orders["order_id"].isin(aug_target_ids)
][[
    "order_id",
    "user_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order"
]].copy()

aug_targets = aug_targets.rename(columns={"order_id": "target_order_id"})

print("Augmented target orders:", len(aug_targets))
print("Unique users:", aug_targets["user_id"].nunique())
print("Order number range:",
      aug_targets["order_number"].min(),
      "to",
      aug_targets["order_number"].max())

Augmented target orders: 100000
Unique users: 55440
Order number range: 4 to 98


In [ ]:
import pandas as pd
import numpy as np
import os
import gc

# Load a manageable sample from the saved augmented training data
sample_parts = []

for f in aug_files:
    part = pd.read_parquet(f)
    sample_parts.append(part)

    if sum(len(x) for x in sample_parts) >= 100000:
        break

aug_model_sample = pd.concat(sample_parts, ignore_index=True).head(100000)

# Add target-order information
aug_model_sample = aug_model_sample.merge(
    aug_targets,
    on="target_order_id",
    how="left"
)

# Product metadata
products_path = f"{base_path}/Datasets/products.csv"
products = pd.read_csv(
    products_path,
    usecols=["product_id", "aisle_id", "department_id"]
)

aug_model_sample = aug_model_sample.merge(
    products,
    on="product_id",
    how="left"
)

print("Sample shape:", aug_model_sample.shape)
print("Target orders:", aug_model_sample["target_order_id"].nunique())
print("Positive rows:", int(aug_model_sample["in_next_basket"].sum()))
print("Positive rate:",
      round(aug_model_sample["in_next_basket"].mean() * 100, 2), "%")
print("Missing values:", int(aug_model_sample.isna().sum().sum()))

del sample_parts, products
gc.collect()

Sample shape: (100000, 11)
Target orders: 4967
Positive rows: 4771
Positive rate: 4.77 %
Missing values: 7559


150

In [ ]:
missing = aug_model_sample.isna().sum()

print("Missing values by column:")
print(missing[missing > 0])

print("\nRows with missing values:",
      aug_model_sample.isna().any(axis=1).sum())

Missing values by column:
target_user_id    7559
dtype: int64

Rows with missing values: 7559


In [ ]:
aug_model_sample = aug_model_sample.drop(columns=["target_user_id"])

aug_model_sample = aug_model_sample.rename(
    columns={"user_id": "target_user_id"}
)

print("Shape:", aug_model_sample.shape)
print("Missing values:", int(aug_model_sample.isna().sum().sum()))
print("Target users:", aug_model_sample["target_user_id"].nunique())
print("Positive rate:",
      round(aug_model_sample["in_next_basket"].mean() * 100, 2), "%")

Shape: (100000, 10)
Missing values: 0
Target users: 2727
Positive rate: 4.77 %


In [ ]:
print("AUGMENTATION ROBUSTNESS SUMMARY")
print("-" * 40)

print("Original candidate-training rows:", 74_633_064)
print("Augmented training rows:", 16_833_417)

print("Original positive rate:", "4.80%")
print("Augmented positive rate:",
      f"{aug_model_sample['in_next_basket'].mean() * 100:.2f}%")

print("Augmented target orders:", 100_000)
print("Augmented users:", 55_440)

print("Sample target orders:", aug_model_sample["target_order_id"].nunique())
print("Sample users:", aug_model_sample["target_user_id"].nunique())

print("\nAugmented candidate coverage of actual basket:", "79.8%")
print("Target orders with candidates:", "100%")

AUGMENTATION ROBUSTNESS SUMMARY
----------------------------------------
Original candidate-training rows: 74633064
Augmented training rows: 16833417
Original positive rate: 4.80%
Augmented positive rate: 4.77%
Augmented target orders: 100000
Augmented users: 55440
Sample target orders: 4967
Sample users: 2727

Augmented candidate coverage of actual basket: 79.8%
Target orders with candidates: 100%


In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Original candidate-training rows",
        "Augmented training rows",
        "Original positive rate",
        "Augmented positive rate",
        "Augmented target orders",
        "Augmented users",
        "Candidate coverage of actual basket",
        "Target orders with candidates"
    ],
    "value": [
        74633064,
        16833417,
        4.80,
        4.77,
        100000,
        55440,
        79.8,
        100.0
    ]
})

summary_path = f"{base_path}/Models/Model_Outputs/augmentation_robustness_summary.csv"

summary.to_csv(summary_path, index=False)

print("Notebook 8 COMPLETE")
print("Saved:", summary_path)
print("\n", summary.to_string(index=False))


Notebook 8 COMPLETE
Saved: /content/drive/MyDrive/ML_Market_Basket_Analysis/04_Results/augmentation_robustness_summary.csv

                              metric       value
   Original candidate-training rows 74633064.00
            Augmented training rows 16833417.00
             Original positive rate        4.80
            Augmented positive rate        4.77
            Augmented target orders   100000.00
                    Augmented users    55440.00
Candidate coverage of actual basket       79.80
      Target orders with candidates      100.00
